In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
import os
import warnings

warnings.filterwarnings("ignore")

1. 다음은 인도 벵갈루루의 집값 데이터이다. 데이터는 집의 특징을 나타내는 입력변수들과 집값의 출력변수로 구성되어 있다.
데이터 컬럼 정의서는 아래와 같을 때, 선형 회귀 분석을 수행하시오.

In [3]:
url = "https://raw.githubusercontent.com/algoboni/pythoncodebook1-1/main/practice8_BHP2.csv"

df = pd.read_csv(url)
df

,area_type,availability,size,total_sqft,bath,balcony,price
0,Super,0,3,1056.0,2,1,39.07
1,Plot,1,6,2600.0,5,3,120.00
2,Super,1,5,1521.0,3,1,95.00
3,Super,1,3,1170.0,2,1,38.00
4,Plot,1,6,2785.0,5,3,295.00
...,...,...,...,...,...,...,...
7490,Super,1,5,1345.0,2,1,57.00
7491,Super,1,5,1715.0,3,3,112.00
7492,Built-up,1,8,3453.0,4,0,231.00
7493,Built-up,1,3,1141.0,2,1,60.00


In [4]:
### 양적변수 : size, total_sqrt, bath, balcony
### 질적변수 : area_type, availabilty, 

### 양적변수 스케일링
from sklearn.preprocessing import StandardScaler

ss = StandardScaler()
df[["size", "total_sqft", "bath", "balcony"]] = ss.fit_transform(df[["size", "total_sqft", "bath", "balcony"]])
df

,area_type,availability,size,total_sqft,bath,balcony,price
0,Super,0,-0.720924,-0.478974,-0.520148,-0.807622,39.07
1,Plot,1,1.421269,1.174833,2.884665,1.756175,120.00
2,Super,1,0.707205,0.019096,0.614790,-0.807622,95.00
3,Super,1,-0.720924,-0.356867,-0.520148,-0.807622,38.00
4,Plot,1,1.421269,1.372990,2.884665,1.756175,295.00
...,...,...,...,...,...,...,...
7490,Super,1,0.707205,-0.169421,-0.520148,-0.807622,57.00
7491,Super,1,0.707205,0.226893,0.614790,1.756175,112.00
7492,Built-up,1,2.849398,2.088497,1.749727,-2.089520,231.00
7493,Built-up,1,-0.720924,-0.387929,-0.520148,-0.807622,60.00


In [5]:
### 데이터 분할

train = df.iloc[:int(len(df)*0.8),:]
test = df.iloc[int(len(df)*0.8):,:]

In [6]:
from statsmodels.api import OLS

formula = "price ~ C(area_type) + C(availability) + size + total_sqft + bath + balcony"

model = OLS.from_formula(formula, data=train).fit()

model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  price   R-squared:                       0.486
Model:                            OLS   Adj. R-squared:                  0.485
Method:                 Least Squares   F-statistic:                     707.0
Date:                Sat, 23 Aug 2025   Prob (F-statistic):               0.00
Time:                        17:15:32   Log-Likelihood:                -34539.
No. Observations:                5996   AIC:                         6.910e+04
Df Residuals:                    5987   BIC:                         6.916e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
==========================================================================================
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 94.8094      3.367     28.155      0.000      88.208     101.411
C(area_type)[T.Carpet]     0.1490     12.032      0.012      0.990     -23.438      23.736
C(area_type)[T.Plot]      83.4291      6.076     13.732      0.000      71.519      95.339
C(area_type)[T.Super]      0.5621      2.790      0.201      0.840      -4.907       6.031
C(availability)[T.1]      -1.9362      2.424     -0.799      0.424      -6.688       2.816
size                      -0.9472      1.739     -0.545      0.586      -4.357       2.463
total_sqft                45.5068      1.207     37.692      0.000      43.140      47.874
bath                      29.8825      1.828     16.351      0.000      26.300      33.465
balcony                   -0.6944      1.079     -0.644      0.520      -2.809       1.420
==============================================================================
Omnibus:                     8595.752   Durbin-Watson:                   2.053
Prob(Omnibus):                  0.000   Jarque-Bera (JB):         18500905.254
Skew:                           7.733   Prob(JB):                         0.00
Kurtosis:                     274.687   Cond. No.                         19.5
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [7]:
### 예측 및 평가
from sklearn.metrics import r2_score

pred = model.predict(test)

r2 = r2_score(test["price"], pred)
r2

0.6808230252395896

### 2. 앞선 모델에서 변수 area_type과 total_sqft의 교호작용항을 추가하여 회귀분석을 수행하시오.

In [8]:
from statsmodels.api import OLS

formula = "price ~ C(area_type)*total_sqft + C(availability) + size + bath + balcony"

model = OLS.from_formula(formula, data=train).fit()

model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  price   R-squared:                       0.546
Model:                            OLS   Adj. R-squared:                  0.545
Method:                 Least Squares   F-statistic:                     653.5
Date:                Sat, 23 Aug 2025   Prob (F-statistic):               0.00
Time:                        17:15:32   Log-Likelihood:                -34167.
No. Observations:                5996   AIC:                         6.836e+04
Df Residuals:                    5984   BIC:                         6.844e+04
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
=====================================================================================================
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                            94.9400      3.166     29.983      0.000      88.733     101.148
C(area_type)[T.Carpet]               -6.7310     13.030     -0.517      0.605     -32.275      18.813
C(area_type)[T.Plot]                  5.1478      6.512      0.790      0.429      -7.619      17.914
C(area_type)[T.Super]                 0.4550      2.624      0.173      0.862      -4.689       5.598
C(availability)[T.1]                 -2.0662      2.280     -0.906      0.365      -6.536       2.404
total_sqft                           21.3217      1.702     12.529      0.000      17.986      24.658
C(area_type)[T.Carpet]:total_sqft     6.9242     15.512      0.446      0.655     -23.485      37.333
C(area_type)[T.Plot]:total_sqft      84.7642      3.025     28.018      0.000      78.833      90.695
C(area_type)[T.Super]:total_sqft     21.1007      2.148      9.822      0.000      16.889      25.312
size                                  1.5852      1.648      0.962      0.336      -1.645       4.815
bath                                 29.8757      1.737     17.198      0.000      26.470      33.281
balcony                              -1.2443      1.015     -1.226      0.220      -3.233       0.745
==============================================================================
Omnibus:                     9359.188   Durbin-Watson:                   2.035
Prob(Omnibus):                  0.000   Jarque-Bera (JB):         19094766.973
Skew:                           9.324   Prob(JB):                         0.00
Kurtosis:                     278.830   Cond. No.                         32.0
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [9]:
### 예측 및 평가
from sklearn.metrics import r2_score

pred = model.predict(test)

r2 = r2_score(test["price"], pred)
r2

0.6733603579792119

In [10]:
### 3. 앞선 모델에서 변수 total_sqft의 이차항을 추가하여 회귀분석을 수행하시오.
from statsmodels.api import OLS

formula = "price ~ C(area_type)*total_sqft + I(total_sqft**2) + C(availability) + size + bath + balcony"

model = OLS.from_formula(formula, data=train).fit()

model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  price   R-squared:                       0.652
Model:                            OLS   Adj. R-squared:                  0.651
Method:                 Least Squares   F-statistic:                     933.0
Date:                Sat, 23 Aug 2025   Prob (F-statistic):               0.00
Time:                        17:15:33   Log-Likelihood:                -33370.
No. Observations:                5996   AIC:                         6.677e+04
Df Residuals:                    5983   BIC:                         6.685e+04
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
=====================================================================================================
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                            99.3260      2.775     35.798      0.000      93.887     104.765
C(area_type)[T.Carpet]               -0.8108     11.411     -0.071      0.943     -23.180      21.558
C(area_type)[T.Plot]                 35.8591      5.748      6.239      0.000      24.591      47.127
C(area_type)[T.Super]                 1.3525      2.298      0.589      0.556      -3.152       5.857
C(availability)[T.1]                 -2.7567      1.997     -1.381      0.167      -6.671       1.157
total_sqft                          112.2229      2.600     43.170      0.000     107.127     117.319
C(area_type)[T.Carpet]:total_sqft   -35.3904     13.619     -2.599      0.009     -62.089      -8.692
C(area_type)[T.Plot]:total_sqft      26.2257      2.983      8.791      0.000      20.378      32.074
C(area_type)[T.Super]:total_sqft     15.1207      1.886      8.016      0.000      11.423      18.819
I(total_sqft ** 2)                   -3.0798      0.072    -42.676      0.000      -3.221      -2.938
size                                -13.8969      1.488     -9.342      0.000     -16.813     -10.981
bath                                 -1.6367      1.691     -0.968      0.333      -4.951       1.678
balcony                              -3.2974      0.890     -3.706      0.000      -5.042      -1.553
==============================================================================
Omnibus:                    10356.631   Durbin-Watson:                   2.005
Prob(Omnibus):                  0.000   Jarque-Bera (JB):         23863012.328
Skew:                          11.628   Prob(JB):                         0.00
Kurtosis:                     311.180   Cond. No.                         465.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [11]:
### 예측 및 평가
from sklearn.metrics import r2_score

pred = model.predict(test)

r2 = r2_score(test["price"], pred)
r2

0.7503280240157819

In [12]:
### 4. 앞선 모델에서 변수 선택 과정을 거쳐 최종 모델을 완성하고 결과를 해석하시오.

In [20]:
from statsmodels.api import OLS
import numpy as np

### 후진 제거법
def backward_method(data, target_col, input_cols):

    best_aic = np.inf
    best_model = None
    best_cols = input_cols.copy()
    
    while True:
        aic_with_candidates = []
        
        for col in best_cols:
            candidates = best_cols.copy()
            candidates.remove(col)
            formula = f"{target_col} ~ {' + '.join(candidates)}"
            model = OLS.from_formula(formula, data=data).fit()
            aic_with_candidates.append((model.aic, col, model, candidates))
        
        # 최소 AIC 확인
        aic_with_candidates.sort(key=lambda x: x[0])
        best_candidate_aic, removed_var, candidate_model, candidate_cols = aic_with_candidates[0]
        
        if best_candidate_aic < best_aic:
            best_aic = best_candidate_aic
            best_model = candidate_model
            best_cols = candidate_cols
        else:
            break
    
    return best_model, best_cols

In [21]:
### 독립변수 컬럼명 리스트
input_cols = ["C(area_type):total_sqft", "C(area_type)", "total_sqft", "I(total_sqft**2)", 
              "C(availability)", "size", "bath", "balcony"]

### 후진 제거법
best_model, cols = backward_method(
                                    data=train,                ## 원본 데이터
                                    target_col="price",        #  종속변수 컬럼명
                                    input_cols=input_cols      #  독립변수 컬럼명 리스트 
                                   )

cols

['C(area_type):total_sqft',
 'C(area_type)',
 'total_sqft',
 'I(total_sqft**2)',
 'size',
 'balcony']

In [15]:
import numpy as np
from statsmodels.api import OLS

### 전진 선택법
def forward_method(data, target_col, input_cols):

    best_aic = np.inf
    best_model = None
    best_cols = []

    while True:
        aic_with_candidates = []

        # 후보 변수 중 아직 선택되지 않은 변수
        remaining_cols = list(set(input_cols) - set(best_cols))

        for col in remaining_cols:
            current_cols = best_cols + [col]                       # 항상 리스트를 만들어 join
            formula = f"{target_col} ~ {' + '.join(current_cols)}"
            model = OLS.from_formula(formula, data=data).fit()
            aic_with_candidates.append((model.aic, col, model))

        if not aic_with_candidates:
            break

        # 최소 AIC 후보 선택
        aic_with_candidates.sort(key=lambda x: x[0])
        best_candidate_aic, best_candidate_col, candidate_model = aic_with_candidates[0]

        # AIC가 개선되면 변수 추가
        if best_candidate_aic < best_aic:
            best_cols.append(best_candidate_col)
            best_model = candidate_model
            best_aic = best_candidate_aic
        else:
            break

    return best_model, best_cols

In [16]:
### 독립변수 컬럼명 리스트
input_cols = ["C(area_type):total_sqft", "C(area_type)", "total_sqft", "I(total_sqft**2)", 
              "C(availability)", "size", "bath", "balcony"]

### 전진 선택법
best_model, cols = forward_method(
                                  data=train,               ## 원본 데이터
                                  target_col="price",       # 종속변수 컬럼명
                                  input_cols=input_cols     # 독립변수 컬럼명 리스트
                                 )
cols

['C(area_type):total_sqft',
 'I(total_sqft**2)',
 'size',
 'C(area_type)',
 'balcony']

In [17]:
from statsmodels.api import OLS

### 단계적 선택법
def stepwise_method(data, target_col, input_cols):
    best_aic = np.inf
    best_model = None
    best_cols = []

    while True:
        changed = False
        aic_with_candidates = []

        # ----- (1) 전진 선택 단계 -----
        remaining_cols = list(set(input_cols) - set(best_cols))
        for col in remaining_cols:
            current_cols = best_cols + [col]
            formula = f"{target_col} ~ {' + '.join(current_cols)}"
            model = OLS.from_formula(formula, data=data).fit()
            aic_with_candidates.append((model.aic, col, model))

        if aic_with_candidates:
            aic_with_candidates.sort(key=lambda x: x[0])
            best_candidate_aic, best_candidate_col, \
                                         candidate_model = aic_with_candidates[0]

            if best_candidate_aic < best_aic:
                best_aic = best_candidate_aic
                best_model = candidate_model
                best_cols.append(best_candidate_col)
                changed = True

        # ----- (2) 후진 제거 단계 -----
        while True:
            aic_with_candidates = []
            for col in best_cols:
                candidates = best_cols.copy()
                candidates.remove(col)
                if not candidates:   # 변수가 다 제거되면 중단
                    continue
                formula = f"{target_col} ~ {' + '.join(candidates)}"
                model = OLS.from_formula(formula, data=data).fit()
                aic_with_candidates.append((model.aic, col, model, candidates))

            if not aic_with_candidates:
                break

            aic_with_candidates.sort(key=lambda x: x[0])
            best_candidate_aic, removed_var, \
                         candidate_model, candidate_cols = aic_with_candidates[0]

            if best_candidate_aic < best_aic:
                best_aic = best_candidate_aic
                best_model = candidate_model
                best_cols = candidate_cols
                changed = True
            else:
                break

        # ----- (3) 종료 조건 -----
        if not changed:
            break

    return best_model, best_cols

In [19]:
### 독립변수 컬럼명 리스트
input_cols = ["C(area_type):total_sqft", "C(area_type)", "total_sqft", "I(total_sqft**2)", 
              "C(availability)", "size", "bath", "balcony"]

best_model, cols = stepwise_method(data=train,             ## 원본 데이터
                                   target_col="price",     #  종속변수 컬럼명
                                   input_cols=input_cols   #  독립변수 컬럼명 리스트
                                   )
cols

['C(area_type):total_sqft',
 'I(total_sqft**2)',
 'size',
 'C(area_type)',
 'balcony']